# Smart Hospitality Management System
## Module 4: Food Image Classification

End-to-end pipeline: data preparation -> EDA -> preprocessing -> MobileNetV2 transfer learning.

**Requires TensorFlow + a downloaded Food-101 subset (or similar food image dataset organized in class subfolders). Run in Google Colab with GPU runtime for reasonable training time.**

**Note:** Section 4 (model building) loads its own train/val/test datasets and is fully self-contained - you can run it on its own after Section 1 (data preparation) has created the `data_split/` folder, even in a fresh kernel session. Section 3 is included for inspecting the pipeline/augmentation but isn't required before Section 4.

## 0. Setup

In [ ]:
# Run this first to create the required folders
import os
os.makedirs('plots', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data_split', exist_ok=True)

# Also make sure your unzipped Food-101 subset is uploaded/mounted here as 'food_subset/'
# with one folder per class, e.g.:
#   food_subset/pizza/*.jpg
#   food_subset/sushi/*.jpg
#   ...


## 1. Data Preparation & Cleaning

In [ ]:
"""
Module 4: Food Image Classification
Step 1: Data Preparation & Cleaning

>>> RUN IN GOOGLE COLAB OR LOCALLY WITH TENSORFLOW INSTALLED <<<
(pip install tensorflow pillow)

Expected input folder structure (after you download your Food-101 subset
and unzip it):

    food_subset/
    ├── pizza/
    │   ├── img1.jpg
    │   └── ...
    ├── sushi/
    │   └── ...
    └── ... (one folder per class)

This script:
  1. Validates every image (drops corrupted/unreadable files - common in
     large scraped datasets like Food-101)
  2. Reports class balance
  3. Splits into train/val/test folders (70/15/15, stratified per class)
     using a copy-based split so the original data stays untouched
"""
import os
import shutil
from pathlib import Path
from PIL import Image
import random

random.seed(42)

# ------------------------------------------------------------------
# EDIT THIS to point at your downloaded & unzipped dataset folder
# ------------------------------------------------------------------
SOURCE_DIR = 'food_subset'          # folder containing one subfolder per class
OUTPUT_DIR = 'data_split'           # where train/val/test will be created

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# ------------------------------------------------------------------
# 1. Discover classes
# ------------------------------------------------------------------
classes = sorted([d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d))])
print(f"Found {len(classes)} classes: {classes}")

# ------------------------------------------------------------------
# 2. Validate images (drop corrupted / unreadable / non-image files)
# ------------------------------------------------------------------
valid_files = {}
total_found, total_valid, total_corrupt = 0, 0, 0

for cls in classes:
    cls_dir = os.path.join(SOURCE_DIR, cls)
    files = os.listdir(cls_dir)
    total_found += len(files)
    good = []
    for fname in files:
        fpath = os.path.join(cls_dir, fname)
        try:
            with Image.open(fpath) as img:
                img.verify()   # checks the file is a valid, non-truncated image
            good.append(fname)
        except Exception:
            total_corrupt += 1
    valid_files[cls] = good
    total_valid += len(good)
    print(f"  {cls}: {len(good)} valid / {len(files)} total")

print(f"\nTotal images found: {total_found}")
print(f"Valid images: {total_valid}")
print(f"Corrupted/unreadable (dropped): {total_corrupt}")

# ------------------------------------------------------------------
# 3. Stratified train/val/test split (copy files into new folder structure)
# ------------------------------------------------------------------
for split in ['train', 'val', 'test']:
    for cls in classes:
        os.makedirs(os.path.join(OUTPUT_DIR, split, cls), exist_ok=True)

split_counts = {'train': 0, 'val': 0, 'test': 0}
for cls, files in valid_files.items():
    random.shuffle(files)
    n = len(files)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    splits = {
        'train': files[:n_train],
        'val': files[n_train:n_train + n_val],
        'test': files[n_train + n_val:]
    }
    for split, split_files in splits.items():
        for fname in split_files:
            src = os.path.join(SOURCE_DIR, cls, fname)
            dst = os.path.join(OUTPUT_DIR, split, cls, fname)
            shutil.copy2(src, dst)
        split_counts[split] += len(split_files)

print(f"\nSplit sizes: {split_counts}")
print(f"Data prepared at -> {OUTPUT_DIR}/{{train,val,test}}/<class>/")


## 2. Exploratory Data Analysis

In [ ]:
"""
Module 4: Food Image Classification
Step 2: Exploratory Data Analysis

>>> RUN IN GOOGLE COLAB OR LOCALLY WITH TENSORFLOW INSTALLED <<<
"""
import os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import random

random.seed(42)
DATA_DIR = 'data_split/train'
classes = sorted(os.listdir(DATA_DIR))

# ------------------------------------------------------------------
# 1. Class distribution (train split)
# ------------------------------------------------------------------
counts = {cls: len(os.listdir(os.path.join(DATA_DIR, cls))) for cls in classes}

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(counts.keys(), counts.values(), color='#4C72B0')
ax.set_title('Training Image Count per Class')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/01_class_distribution.png')
plt.close()

print("Images per class (train):")
for cls, c in counts.items():
    print(f"  {cls}: {c}")

# ------------------------------------------------------------------
# 2. Sample image grid (one per class)
# ------------------------------------------------------------------
n_classes = len(classes)
cols = 4
rows = (n_classes + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = axes.flatten()

for i, cls in enumerate(classes):
    cls_dir = os.path.join(DATA_DIR, cls)
    sample_file = random.choice(os.listdir(cls_dir))
    img = Image.open(os.path.join(cls_dir, sample_file))
    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=10)
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig('plots/02_sample_images.png')
plt.close()

# ------------------------------------------------------------------
# 3. Image size / aspect ratio distribution (sampled, for speed)
# ------------------------------------------------------------------
widths, heights = [], []
sample_per_class = 50
for cls in classes:
    cls_dir = os.path.join(DATA_DIR, cls)
    files = os.listdir(cls_dir)
    sample = random.sample(files, min(sample_per_class, len(files)))
    for fname in sample:
        with Image.open(os.path.join(cls_dir, fname)) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(widths, bins=30, color='#55A868', alpha=0.7, label='Width')
axes[0].hist(heights, bins=30, color='#C44E52', alpha=0.7, label='Height')
axes[0].set_title('Image Dimension Distribution (sampled)')
axes[0].legend()

aspect_ratios = np.array(widths) / np.array(heights)
axes[1].hist(aspect_ratios, bins=30, color='#8172B3')
axes[1].set_title('Aspect Ratio Distribution (width/height)')
plt.tight_layout()
plt.savefig('plots/03_image_dimensions.png')
plt.close()

print(f"\nImage width range: {min(widths)}-{max(widths)}, avg {np.mean(widths):.0f}")
print(f"Image height range: {min(heights)}-{max(heights)}, avg {np.mean(heights):.0f}")
print("\nAll images will be resized to a fixed size (e.g. 224x224) in preprocessing,")
print("so varying source dimensions are not a concern.")
print("\nPlots saved to plots/ directory")


## 3. Preprocessing (tf.data pipelines + augmentation) - optional, for inspection only

In [ ]:
"""
Module 4: Food Image Classification
Step 3: Preprocessing (tf.data pipelines + augmentation)

>>> RUN IN GOOGLE COLAB OR LOCALLY WITH TENSORFLOW INSTALLED <<<
"""
import tensorflow as tf
from tensorflow.keras import layers

IMG_SIZE = (224, 224)   # standard input size for MobileNetV2
BATCH_SIZE = 32
DATA_DIR = 'data_split'

# ------------------------------------------------------------------
# 1. Load datasets from directory structure
# ------------------------------------------------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/train', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=True, seed=42
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/val', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/test', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False
)

class_names = train_ds.class_names
print(f"Classes ({len(class_names)}): {class_names}")

import json
with open('models/class_names.json', 'w') as f:
    json.dump(class_names, f)

# ------------------------------------------------------------------
# 2. Data augmentation (applied only during training, inside the model
#    graph so it runs on GPU and doesn't need separate preprocessing)
# ------------------------------------------------------------------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name='data_augmentation')

# ------------------------------------------------------------------
# 3. Performance: prefetch + cache
# ------------------------------------------------------------------
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("\ntf.data pipelines ready: train_ds, val_ds, test_ds")
print("Note: MobileNetV2's own preprocess_input (rescale to [-1,1]) is applied")
print("inside the model itself in the next script, not here.")


## 4. Model Building — MobileNetV2 Transfer Learning (self-contained: loads its own data)

In [ ]:
"""
Module 4: Food Image Classification
Step 4: Model Building — MobileNetV2 Transfer Learning

>>> RUN IN GOOGLE COLAB (GPU RECOMMENDED) OR LOCALLY WITH TENSORFLOW <<<

Two-phase training strategy (standard transfer learning best practice):
  Phase 1: freeze the pretrained MobileNetV2 base, train only the new
           classification head (fast, prevents destroying pretrained
           weights with large early gradients)
  Phase 2: unfreeze the top layers of the base and fine-tune with a very
           low learning rate (lets the model adapt pretrained ImageNet
           features to food-specific textures/shapes)

Run 01_data_preparation.py, 02_eda.py, 03_preprocessing.py first (or paste
their contents into the same notebook session) so train_ds/val_ds/test_ds
and class_names exist before running this.
"""
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, classification_report

tf.random.set_seed(42)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = 'data_split'

# ------------------------------------------------------------------
# 0. Load the datasets (self-contained - doesn't assume 03_preprocessing.py
#    variables are still in memory, e.g. if running this script on its own
#    or after a kernel restart)
# ------------------------------------------------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/train', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=True, seed=42
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/val', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/test', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"Classes ({NUM_CLASSES}): {class_names}")

os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)
with open('models/class_names.json', 'w') as f:
    json.dump(class_names, f)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# ------------------------------------------------------------------
# 1. Build the model: MobileNetV2 base + custom classification head
# ------------------------------------------------------------------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name='data_augmentation')

base_model = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False   # Phase 1: freeze the pretrained base

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)             # MobileNetV2's expected [-1, 1] scaling
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ------------------------------------------------------------------
# 2. Phase 1: train the classification head only
# ------------------------------------------------------------------
callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

print("\n=== Phase 1: Training classification head (base frozen) ===")
history1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks_phase1)

# ------------------------------------------------------------------
# 3. Phase 2: unfreeze top layers of the base and fine-tune
# ------------------------------------------------------------------
base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 30   # unfreeze only the last ~30 layers
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
]

print("\n=== Phase 2: Fine-tuning top layers of MobileNetV2 ===")
history2 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks_phase2)

# ------------------------------------------------------------------
# 4. Combine training histories & plot curves
# ------------------------------------------------------------------
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
phase1_epochs = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(acc, label='Train Acc')
axes[0].plot(val_acc, label='Val Acc')
axes[0].axvline(phase1_epochs, color='gray', linestyle='--', label='Fine-tuning starts')
axes[0].set_title('Accuracy'); axes[0].legend()

axes[1].plot(loss, label='Train Loss')
axes[1].plot(val_loss, label='Val Loss')
axes[1].axvline(phase1_epochs, color='gray', linestyle='--', label='Fine-tuning starts')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig('plots/04_training_curves.png')
plt.close()

# ------------------------------------------------------------------
# 5. Evaluate on the held-out test set
# ------------------------------------------------------------------
test_loss, test_acc = model.evaluate(test_ds)
print(f"\n--- Test Set Performance ---")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss:     {test_loss:.4f}")

# ------------------------------------------------------------------
# 6. Detailed per-class metrics + confusion matrix
# ------------------------------------------------------------------
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\nPer-class report:")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_names, yticklabels=class_names)
ax.set_title('Confusion Matrix - Food Classification')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/05_confusion_matrix.png')
plt.close()

# ------------------------------------------------------------------
# 7. Save the model
# ------------------------------------------------------------------
model.save('models/mobilenetv2_food_classifier.keras')
print("\nModel saved -> models/mobilenetv2_food_classifier.keras")
